In [ ]:
!apt-get -y update
!apt-get -y install mafft fasttree
!pip -q install biopython ete4 scipy matplotlib

In [ ]:
FL_faa_path = "aor0-5_SM00066_FL.faa"
Domain_faa_path = "aor0-5_SM00066_domains_2.faa"
Non_Domain_faa_path = "aor0-5_non_SM00066_domains_AO090012000182_fixed.faa"

In [ ]:
import shutil, textwrap, subprocess

In [ ]:
import os
import re
from collections import defaultdict
from Bio import SeqIO

In [ ]:
def normalize_ids(in_faa: str, out_faa: str) -> None:

    counts = defaultdict(int)

    records_out = []
    for rec in SeqIO.parse(in_faa, "fasta"):
        raw = rec.id  # first token
        base = raw.split("-P", 1)[0]  # remove '-P' and everything after it (leave unchanged if absent)

        counts[base] += 1
        new_id = base if counts[base] == 1 else f"{base}_{counts[base]}"

        rec.id = new_id
        rec.name = new_id
        rec.description = ""  # remove extra descriptions and retain only the ID
        records_out.append(rec)

    SeqIO.write(records_out, out_faa, "fasta")

# Determine the input file
if "FL_faa_path" not in globals():
    # When upload() is used, select the first .faa file (edit as needed if multiple files are present)
    faa_candidates = [fn for fn in os.listdir("/content") if fn.endswith((".faa", ".faa.gz", ".fasta", ".fa"))]
    if len(faa_candidates) == 0:
        raise FileNotFoundError("No FASTA/FAA files found in /content. Upload a .faa file first.")
    FL_faa_path = "/content/" + sorted(faa_candidates)[0]

FL_norm_faa = "FL_normalized.faa"
normalize_ids(FL_faa_path, FL_norm_faa)

print("Input :", FL_faa_path)
print("Output:", FL_norm_faa)

# Check that duplicate IDs have been resolved
ids = [r.id for r in SeqIO.parse(FL_norm_faa, "fasta")]
dups = [x for x in set(ids) if ids.count(x) > 1]
print("N sequences:", len(ids))
print("Duplicated IDs after normalization:", dups[:10], "..." if len(dups) > 10 else "")

In [ ]:
FL_aln_faa  = "FL_aligned.faa"
FL_tree_file = "FL_tree.nwk"

# --- MAFFT ---
mafft_bin = shutil.which("mafft")
assert mafft_bin is not None, "mafft not found. Run: !apt-get -y install mafft"

# Equivalent to E-INS-i: intended for sequences with long unalignable regions and large internal gaps
# Use one thread to prioritize reproducibility
cmd = [
    mafft_bin,
    "--amino",
    "--ep", "0",
    "--genafpair",
    "--maxiterate", "1000",
    "--thread", "1",
    FL_norm_faa,
]
with open(FL_aln_faa, "w") as out:
    p = subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)

print("MAFFT return code:", p.returncode)
if p.returncode != 0:
    print(p.stderr[:4000])
    raise RuntimeError("MAFFT failed")

print("Aligned FASTA:", FL_aln_faa, "size:", os.path.getsize(FL_aln_faa))

# --- FastTree ---
fasttree_bin = shutil.which("FastTree") or shutil.which("fasttree")
assert fasttree_bin is not None, "FastTree not found. Run: !apt-get -y install fasttree"

cmd = [fasttree_bin, "-lg", FL_aln_faa]
with open(FL_tree_file, "w") as out:
    p = subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)

print("FastTree return code:", p.returncode)
if p.returncode != 0:
    print(p.stderr[:4000])
    raise RuntimeError("FastTree failed")

print("Tree (Newick):", FL_tree_file, "size:", os.path.getsize(FL_tree_file))


In [ ]:
from google.colab import files
files.download("FL_tree.nwk")

In [ ]:
def normalize_ids_non_DBD(in_faa: str, out_faa: str) -> None:
    """
    - Use the first token of the FASTA header (up to the first space) as the ID
    - Remove '-P' and everything after it
    - If an ID occurs multiple times, append _1, _2, ... in input order
    """
    counts = defaultdict(int)

    records_out = []
    for rec in SeqIO.parse(in_faa, "fasta"):
        raw = rec.id  # first token
        base = raw.split("-P", 1)[0]  # remove '-P' and everything after it (leave unchanged if absent)

        counts[base] += 1
        new_id = base if counts[base] == 1 else f"{base}_{counts[base]}"

        rec.id = new_id
        rec.name = new_id
        rec.description = ""  # remove extra descriptions and retain only the ID
        records_out.append(rec)

    SeqIO.write(records_out, out_faa, "fasta")

# Determine the input file
if "Non_Domain_faa_path" not in globals():
    # When upload() is used, select the first .faa file (edit as needed if multiple files are present)
    faa_candidates = [fn for fn in os.listdir("/content") if fn.endswith((".faa", ".faa.gz", ".fasta", ".fa"))]
    if len(faa_candidates) == 0:
        raise FileNotFoundError("No FASTA/FAA files found in /content. Upload a .faa file first.")
    Non_Domain_faa_path = "/content/" + sorted(faa_candidates)[0]

non_DBD_norm_faa = "non_DBD_normalized.faa"
normalize_ids_non_DBD(Non_Domain_faa_path, non_DBD_norm_faa)

print("Input :", Non_Domain_faa_path)
print("Output:", non_DBD_norm_faa)

# Check that duplicate IDs have been resolved
ids = [r.id for r in SeqIO.parse(non_DBD_norm_faa, "fasta")]
dups = [x for x in set(ids) if ids.count(x) > 1]
print("N sequences:", len(ids))
print("Duplicated IDs after normalization:", dups[:10], "..." if len(dups) > 10 else "")

In [ ]:
non_DBD_aln_faa  = "non_DBD_aligned.faa"
non_DBD_tree_file = "non_DBD_tree.nwk"

# --- MAFFT ---
mafft_bin = shutil.which("mafft")
assert mafft_bin is not None, "mafft not found. Run: !apt-get -y install mafft"

# Equivalent to E-INS-i: intended for sequences with long unalignable regions and large internal gaps
# Use one thread to prioritize reproducibility
cmd = [
    mafft_bin,
    "--amino",
    "--ep", "0",
    "--genafpair",
    "--maxiterate", "1000",
    "--thread", "1",
    non_DBD_norm_faa,
]
with open(non_DBD_aln_faa, "w") as out:
    p = subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)

print("MAFFT return code:", p.returncode)
if p.returncode != 0:
    print(p.stderr[:4000])
    raise RuntimeError("MAFFT failed")

print("Aligned FASTA:", non_DBD_aln_faa, "size:", os.path.getsize(non_DBD_aln_faa))

# --- FastTree ---
fasttree_bin = shutil.which("FastTree") or shutil.which("fasttree")
assert fasttree_bin is not None, "FastTree not found. Run: !apt-get -y install fasttree"

cmd = [fasttree_bin, "-lg", non_DBD_aln_faa]
with open(non_DBD_tree_file, "w") as out:
    p = subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)

print("FastTree return code:", p.returncode)
if p.returncode != 0:
    print(p.stderr[:4000])
    raise RuntimeError("FastTree failed")

print("Tree (Newick):", non_DBD_tree_file, "size:", os.path.getsize(non_DBD_tree_file))


In [ ]:
from google.colab import files
files.download("non_DBD_tree.nwk")

In [ ]:
def Domain_normalize_ids(in_faa: str, out_faa: str) -> None:
    """
    - Use the first token of the FASTA header (up to the first space) as the ID
    - Remove '-P' and everything after it
    - If an ID occurs multiple times, append _1, _2, ... in input order
    """
    counts = defaultdict(int)

    records_out = []
    for rec in SeqIO.parse(in_faa, "fasta"):
        raw = rec.id  # first token
        base = raw.split("-P", 1)[0]  # remove '-P' and everything after it (leave unchanged if absent)

        counts[base] += 1
        new_id = base if counts[base] == 1 else f"{base}_{counts[base]}"

        rec.id = new_id
        rec.name = new_id
        rec.description = ""  # remove extra descriptions and retain only the ID
        records_out.append(rec)

    SeqIO.write(records_out, out_faa, "fasta")

# Determine the input file
if "Domain_faa_path" not in globals():
    # When upload() is used, select the first .faa file (edit as needed if multiple files are present)
    faa_candidates = [fn for fn in os.listdir("/content") if fn.endswith((".faa", ".faa.gz", ".fasta", ".fa"))]
    if len(faa_candidates) == 0:
        raise FileNotFoundError("No FASTA/FAA files found in /content. Upload a .faa file first.")
    Domain_faa_path = "/content/" + sorted(faa_candidates)[0]

Domain_norm_faa = "Domain_normalized.faa"
normalize_ids(Domain_faa_path, Domain_norm_faa)

print("Input :", Domain_faa_path)
print("Output:", Domain_norm_faa)

# Check that duplicate IDs have been resolved
ids = [r.id for r in SeqIO.parse(Domain_norm_faa, "fasta")]
dups = [x for x in set(ids) if ids.count(x) > 1]
print("N sequences:", len(ids))
print("Duplicated IDs after normalization:", dups[:10], "..." if len(dups) > 10 else "")

In [ ]:
Domain_aln_faa  = "Domain_aligned.faa"
Domain_tree_file = "Domain_tree.nwk"

# --- MAFFT ---
mafft_bin = shutil.which("mafft")
assert mafft_bin is not None, "mafft not found. Run: !apt-get -y install mafft"

# Equivalent to E-INS-i: intended for sequences with long unalignable regions and large internal gaps
# Use one thread to prioritize reproducibility
cmd = [
    mafft_bin,
    "--amino",
    "--ep", "0",
    "--genafpair",
    "--maxiterate", "1000",
    "--thread", "1",
    Domain_norm_faa,
]
with open(Domain_aln_faa, "w") as out:
    p = subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)

print("MAFFT return code:", p.returncode)
if p.returncode != 0:
    print(p.stderr[:4000])
    raise RuntimeError("MAFFT failed")

print("Aligned FASTA:", Domain_aln_faa, "size:", os.path.getsize(Domain_aln_faa))

# --- FastTree ---
fasttree_bin = shutil.which("FastTree") or shutil.which("fasttree")
assert fasttree_bin is not None, "FastTree not found. Run: !apt-get -y install fasttree"

cmd = [fasttree_bin, "-lg", Domain_aln_faa]
with open(Domain_tree_file, "w") as out:
    p = subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)

print("FastTree return code:", p.returncode)
if p.returncode != 0:
    print(p.stderr[:4000])
    raise RuntimeError("FastTree failed")

print("Tree (Newick):", Domain_tree_file, "size:", os.path.getsize(Domain_tree_file))


In [ ]:
files.download("Domain_tree.nwk")

Clustering > Ten support-prioritized display groups, coloring, and annotation of focal branches

Under the constraint that ten non-overlapping monophyletic groups cover all leaves, dynamic programming identifies the partition that maximizes the sum of the number of leaves multiplied by the FastTree SH-like local support at the basal branch of each group. This is equivalent to maximizing the leaf-count-weighted mean basal support. Because a single leaf lacks internal basal-branch support, it is assigned a score of zero during optimization and reported as missing in the results table. The number of groups (ten) and the objective function are analytical choices, not an estimate of a natural number of groups from the data.
Values in the Newick files are FastTree SH-like local support values (0-1), not conventional bootstrap proportions.

In [ ]:
!pip -q install ete4

import csv
from pathlib import Path
from typing import Dict, List
from ete4 import Tree

TREE_FILES = {
    "FL": "FL_tree.nwk",
    "non_DBD": "non_DBD_tree.nwk",
    "Domain": "Domain_tree.nwk",
}
K = 10
SUPPORT_LABEL_SIZE = 4.0  # approximately the same size as the iTOL ID labels
SUPPORT_LABEL_POSITION = 0.88  # on the terminal branch immediately inside the ID label

PALETTE = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]

# Retain the focal IDs highlighted in the existing notebook
HIGHLIGHT_IDS = [
    "AO090003001208", "AO090012000267", "AO090003001292",
    "AO090012000035", "AO090038000235", "AO090113000137",
    "AO090012000182", "AO090012000002", "AO090012000002_1",
    "AO090012000002_2", "AO090026000014", "AO090026000614",
    "AO090701000449", "AO090005001236",
]

# Focal IDs for which the nearest branch support is displayed (requested order)
MAJOR_IDS = [
    "AO090003001208", "AO090012000267", "AO090003001292",
    "AO090038000235", "AO090113000137", "AO090701000449",
    "AO090012000035", "AO090026000614", "AO090026000014",
    "AO090012000182", "AO090012000002", "AO090012000002_2",
    "AO090005001236",
]

NEAR_BGC_IDS = ['AO090009000717', 'AO090009000079', 'AO090009000133', 'AO090005001545', 'AO090005001236', 'AO090005001228', 'AO090005001087', 'AO090005001663', 'AO090005000990', 'AO090005000933', 'AO090001000753', 'AO090001000029', 'AO090001000755', 'AO090001000756', 'AO090001000372', 'AO090001000392', 'AO090001000515', 'AO090003000962', 'AO090003001543', 'AO090023000280', 'AO090023000305', 'AO090023000416', 'AO090026000014', 'AO090012000525', 'AO090102000477', 'AO090102000449', 'AO090701000776', 'AO090701000217', 'AO090113000035', 'AO090113000164', 'AO090113000204', 'AO090020000384', 'AO090020000136', 'AO090038000557', 'AO090038000532', 'AO090038000494', 'AO090038000085', 'AO090038000032', 'AO090011000961', 'AO090011000729', 'AO090103000377', 'AO090010000442', 'AO090010000338', 'AO090010000784', 'AO090010000324', 'AO090010000113', 'AO090010000049', 'AO090010000037']

HIGHLIGHT_BG_COLOR = "#fff2a8"
SPECIAL_HIGHLIGHT_BASE_IDS = {"AO090012000182", "AO090012000002"}
SPECIAL_HIGHLIGHT_COLOR = "#f28e8e"
NEAR_BGC_BG_COLOR = "#bfefff"
OVERLAP_BG_COLOR = "#d8b4fe"
FOCAL_TF_ID = "AO090005001236"
FOCAL_TF_COLOR = "#b000b5"
HIGHLIGHT_BRANCH_WIDTH = 4
NORMAL_BRANCH_WIDTH = 1


In [ ]:
def clean_id(x: str) -> str:
    return x.strip().strip("'").strip('"')

def leaves_list(node):
    return list(node.leaves())

def leaf_names(node):
    return [clean_id(x.name) for x in node.leaves()]

def node_support(node):
    """Assign the lowest split priority to the root, leaves, and missing values."""
    value = node.support
    return float(value) if value is not None else -1.0

def make_support_optimized_k_clades(tree, k=10):
    """Use dynamic programming to identify k monophyletic groups that maximize leaf-count-weighted basal support."""
    memo = {}

    def solve(node, n_groups):
        cache_key = (id(node), n_groups)
        if cache_key in memo:
            return memo[cache_key]
        if n_groups == 1:
            # Leaves lack internal branch support and therefore receive a score of zero during optimization.
            support = 0.0 if node.is_leaf else max(node_support(node), 0.0)
            result = (len(node) * support, [node])
            memo[cache_key] = result
            return result
        if node.is_leaf or n_groups < len(node.children):
            memo[cache_key] = None
            return None

        # Allocate at least one group to each child clade.
        dp = {0: (0.0, [])}
        for child in node.children:
            next_dp = {}
            for used, (score_so_far, groups_so_far) in dp.items():
                for take in range(1, n_groups - used + 1):
                    child_result = solve(child, take)
                    if child_result is None:
                        continue
                    score = score_so_far + child_result[0]
                    groups = groups_so_far + child_result[1]
                    previous = next_dp.get(used + take)
                    if previous is None or score > previous[0]:
                        next_dp[used + take] = (score, groups)
            dp = next_dp
        memo[cache_key] = dp.get(n_groups)
        return memo[cache_key]

    result = solve(tree, k)
    if result is None:
        raise RuntimeError(f"Unable to create {k} monophyletic groups covering all leaves")
    objective_score, clades = result
    clades = sorted(clades, key=lambda n: (-len(n), min(leaf_names(n))))
    return clades, objective_score

def build_clusters(tree, k=10):
    clades, objective_score = make_support_optimized_k_clades(tree, k)
    leaf_to_cluster = {}
    for cid, node in enumerate(clades, 1):
        for name in leaf_names(node):
            if name in leaf_to_cluster:
                raise RuntimeError(f"Duplicate assignment: {name}")
            leaf_to_cluster[name] = cid
    all_names = leaf_names(tree)
    if set(leaf_to_cluster) != set(all_names):
        raise RuntimeError("The group assignment contains missing or extra IDs")
    return clades, leaf_to_cluster, objective_score

def matching_leaves(tree, requested_id):
    """Match explicit _1/_2 identifiers exactly; for a base ID, retrieve the unsuffixed and _1/_2 forms."""
    requested_id = clean_id(requested_id)
    if requested_id.endswith(("_1", "_2")):
        return [x for x in tree.leaves() if clean_id(x.name) == requested_id]
    return [x for x in tree.leaves()
            if clean_id(x.name) == requested_id or clean_id(x.name).startswith(requested_id + "_")]

def map_ids(requested_ids, leaf_ids):
    leaf_set = set(leaf_ids)
    mapped = []
    for requested in requested_ids:
        if requested.endswith(("_1", "_2")):
            candidates = [requested]
        else:
            candidates = [requested, requested + "_1", requested + "_2"]
        mapped.extend(x for x in candidates if x in leaf_set)
    return list(dict.fromkeys(mapped))

def node_key(node):
    """Create an iTOL LCA specification using one leaf from each of two different child clades."""
    if node.is_leaf or len(node.children) < 2:
        return None
    left = next(iter(node.children[0].leaves())).name
    right = next(iter(node.children[1].leaves())).name
    return f"{left}|{right}"


In [ ]:
def write_colorstrip(path, leaf_ids, leaf_to_cluster, clades, tree_name):
    legend_labels = []
    for cid, node in enumerate(clades, 1):
        support_text = "NA" if node.is_leaf or node.support is None else f"{float(node.support):.3f}"
        legend_labels.append(f"G{cid} (SH-like={support_text})")
    with open(path, "w", encoding="utf-8") as out:
        out.write("DATASET_COLORSTRIP\nSEPARATOR TAB\n")
        out.write(f"DATASET_LABEL\t{tree_name}_support_prioritized_k{K}\nCOLOR\t#000000\n")
        out.write("STRIP_WIDTH\t25\nMARGIN\t5\nBORDER_WIDTH\t0\n")
        out.write("LEGEND_TITLE\tSupport-prioritized display groups\n")
        out.write("LEGEND_SCALE\t1.2\n")
        out.write("LEGEND_SHAPES\t" + "\t".join(["1"] * K) + "\n")
        out.write("LEGEND_COLORS\t" + "\t".join(PALETTE) + "\n")
        out.write("LEGEND_LABELS\t" + "\t".join(legend_labels) + "\nDATA\n")
        for leaf in leaf_ids:
            cid = leaf_to_cluster[leaf]
            out.write(f"{leaf}\t{PALETTE[cid-1]}\tG{cid}\n")

def highlight_color(leaf, original_set, near_set):
    base = leaf.rsplit("_", 1)[0] if leaf.endswith(("_1", "_2")) else leaf
    if base == FOCAL_TF_ID:
        return FOCAL_TF_COLOR
    if leaf in original_set and leaf in near_set:
        return OVERLAP_BG_COLOR
    if leaf in near_set:
        return NEAR_BGC_BG_COLOR
    if base in SPECIAL_HIGHLIGHT_BASE_IDS:
        return SPECIAL_HIGHLIGHT_COLOR
    return HIGHLIGHT_BG_COLOR

def write_treecolors(path, leaf_ids, leaf_to_cluster, original_ids, near_ids):
    original_set, near_set = set(original_ids), set(near_ids)
    with open(path, "w", encoding="utf-8") as out:
        out.write("TREE_COLORS\nSEPARATOR TAB\nDATA\n")
        for leaf in leaf_ids:
            cid = leaf_to_cluster[leaf]
            group_color = PALETTE[cid-1]
            emphasized = leaf in original_set or leaf in near_set
            if emphasized:
                bg = highlight_color(leaf, original_set, near_set)
                out.write(f"{leaf}\tlabel_background\t{bg}\n")
                out.write(f"{leaf}\tlabel\t{group_color}\tbold\n")
                out.write(f"{leaf}\tbranch\t{bg}\tnormal\t{HIGHLIGHT_BRANCH_WIDTH}\n")
            else:
                out.write(f"{leaf}\tlabel\t{group_color}\tnormal\n")
                out.write(f"{leaf}\tbranch\t{group_color}\tnormal\t{NORMAL_BRANCH_WIDTH}\n")

def write_inner_major_support(path, leaf_support):
    """Place the nearest-branch support for each focal ID horizontally inside its terminal branch."""
    with open(path, "w", encoding="utf-8") as out:
        out.write("DATASET_TEXT\nSEPARATOR TAB\n")
        out.write("DATASET_LABEL\tNearest-branch SH-like support\n")
        out.write("COLOR\t#000000\n")
        out.write("STRAIGHT_LABELS\t1\n")
        out.write("LABELS_BELOW\t0\nVERTICAL_SHIFT\t-2\nDATA\n")
        for leaf, value in leaf_support.items():
            # Use the leaf ID as the key for placement on the terminal branch. A position of 0.88 is near the leaf tip and inside the ID label.
            out.write(
                f"{leaf}\t{value:.3f}\t{SUPPORT_LABEL_POSITION:.2f}\t#000000\tnormal\t"
                f"{SUPPORT_LABEL_SIZE}\t0\n"
            )


In [ ]:
all_reports = []

for tree_name, tree_file in TREE_FILES.items():
    print(f"\n=== {tree_name}: support-prioritized {K}-group partition ===")
    tree = Tree(Path(tree_file).read_text(encoding="utf-8"), parser=0)
    leaf_ids = leaf_names(tree)
    clades, leaf_to_cluster, objective_score = build_clusters(tree, K)
    mapped_highlights = map_ids(HIGHLIGHT_IDS, leaf_ids)
    mapped_near_bgc = map_ids(NEAR_BGC_IDS, leaf_ids)

    selected = {}
    major_leaf_support = {}
    report = []

    def record(feature, node, queried_id="", record_type="", show_on_tree=True):
        support = None if node.is_leaf else node.support
        key = node_key(node)
        row = {
            "tree": tree_name, "record_type": record_type, "feature": feature,
            "n_descendant_leaves": len(node),
            "SH_like_local_support": support,
            "itol_LCA_node": key or "", "queried_id": queried_id,
        }
        report.append(row)
        if show_on_tree and key and support is not None and node is not tree:
            selected[key] = float(support)

    # Basal support values for the ten display groups
    for cid, node in enumerate(clades, 1):
        record(f"G{cid}_basal_branch", node, record_type="display_group", show_on_tree=False)

    # Nearest parent branches of the focal IDs. Record each DBD leaf of AO090012000002 separately in the domain tree.
    already = set()
    for requested_id in MAJOR_IDS:
        found = matching_leaves(tree, requested_id)
        if not found:
            print(f"[WARN] {tree_name}: not found: {requested_id}")
        for leaf in found:
            if leaf.name in already:
                continue
            already.add(leaf.name)
            if leaf.up is not None:
                record(f"{leaf.name}_nearest_internal_branch", leaf.up,
                       queried_id=requested_id, record_type="major_ID")
                if leaf.up.support is not None:
                    major_leaf_support[leaf.name] = float(leaf.up.support)

    write_colorstrip(f"{tree_name}_itol_colorstrip_support_k10.txt",
                     leaf_ids, leaf_to_cluster, clades, tree_name)
    write_treecolors(f"{tree_name}_itol_treecolors_support_k10_highlights.txt",
                     leaf_ids, leaf_to_cluster, mapped_highlights, mapped_near_bgc)
    # Display each focal-ID value on the corresponding terminal branch immediately inside the ID label.
    inner_support_file = f"{tree_name}_itol_major_ID_support_inner.txt"
    write_inner_major_support(inner_support_file, major_leaf_support)

    with open(f"{tree_name}_support_k10_membership.tsv", "w", encoding="utf-8", newline="") as out:
        writer = csv.writer(out, delimiter="\t")
        writer.writerow(["ID", "Cluster"])
        for leaf in leaf_ids:
            writer.writerow([leaf, f"G{leaf_to_cluster[leaf]}"])

    with open(f"{tree_name}_support_k10_and_major_ID_report.tsv", "w", encoding="utf-8", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=report[0].keys(), delimiter="\t")
        writer.writeheader(); writer.writerows(report)

    all_reports.extend(report)
    print(f"Weighted support objective: {objective_score:.3f}")
    print("Group  n_leaves  basal_support")
    for cid, node in enumerate(clades, 1):
        print(f"G{cid:<2}    {len(node):<8}  {node.support}")
    print("Major IDs: leaf  n_descendants  nearest_support")
    for row in report:
        if row["record_type"] == "major_ID":
            print(row["feature"], row["n_descendant_leaves"], row["SH_like_local_support"])
    print("iTOL inner support labels:", inner_support_file)

print("\nNote: Ten is the specified number of display groups; it is not an estimate of the optimal number of groups from support values.")
print("For each focal ID, display only the numeric support value in black, horizontally inside the corresponding terminal branch.")


In [ ]:
FOCAL_GROUPS = {
    "AmyR_XlnR_AraR": {
        "AO090003001208": "AmyR",
        "AO090012000267": "XlnR",
        "AO090003001292": "AraR",
    },
    "KojR_AtrR": {
        "AO090113000137": "KojR",
        "AO090026000614": "AtrR",
    },
}

for filename in TREE_FILES.values():
    if not Path(filename).is_file():
        raise FileNotFoundError(filename)


def clean_id(value):
    return str(value).strip().strip("'").strip('"')


def matching_leaves(tree, base_id):
    """Retrieve leaves that exactly match the base ID and leaves assigned _1/_2 during normalization."""
    return [
        leaf for leaf in tree.leaves()
        if clean_id(leaf.name) == base_id
        or clean_id(leaf.name).startswith(base_id + "_")
    ]


def itol_lca_key(node):
    """Return two leaves from different child branches that can specify the MRCA in iTOL."""
    if node.is_leaf or len(node.children) < 2:
        return ""
    left = clean_id(next(iter(node.children[0].leaves())).name)
    right = clean_id(next(iter(node.children[1].leaves())).name)
    return f"{left}|{right}"


rows = []

for tree_label, filename in TREE_FILES.items():
    # Read numeric internal labels in the FastTree Newick file as support values.
    tree = Tree(Path(filename).read_text(encoding="utf-8").strip(), parser=0)
    for group_name, requested in FOCAL_GROUPS.items():
        matched_nodes = []

        for base_id, symbol in requested.items():
            found = matching_leaves(tree, base_id)
            if not found:
                raise ValueError(
                    f"{tree_label}: {symbol} ({base_id})  is absent from the tree"
                )
            matched_nodes.extend(found)

        # Do not pass the same leaf twice.
        matched_nodes = list({clean_id(x.name): x for x in matched_nodes}.values())
        mrca = tree.common_ancestor(matched_nodes)
        descendant_ids = sorted(clean_id(x.name) for x in mrca.leaves())
        matched_names = sorted(clean_id(x.name) for x in matched_nodes)
        matched_set = set(matched_names)
        extra_ids = sorted(set(descendant_ids) - matched_set)

        is_root = mrca is tree
        support = None if is_root or mrca.support is None else float(mrca.support)

        rows.append({
            "tree": tree_label,
            "newick_file": filename,
            "focal_group": group_name,
            "requested_genes": ";".join(
                f"{symbol}={base_id}" for base_id, symbol in requested.items()
            ),
            "matched_tree_leaves": ";".join(matched_names),
            "mrca_is_root": is_root,
            "SH_like_local_support": "" if support is None else f"{support:.3f}",
            "n_queried_leaves": len(matched_names),
            "n_mrca_descendant_leaves": len(descendant_ids),
            "n_extra_descendant_leaves": len(extra_ids),
            "forms_exclusive_clade": len(extra_ids) == 0,
            "itol_LCA_node": itol_lca_key(mrca),
            "extra_descendant_ids": ";".join(extra_ids),
            "all_mrca_descendant_ids": ";".join(descendant_ids),
        })


out_tsv = Path("focal_group_MRCA_support.tsv")
with out_tsv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]), delimiter="\t")
    writer.writeheader()
    writer.writerows(rows)

print("\nMRCA summary")
print("tree\tgroup\tsupport\tn_descendants\tn_extra\texclusive\troot")
for row in rows:
    support = row["SH_like_local_support"] or "NA"
    print(
        f'{row["tree"]}\t{row["focal_group"]}\t{support}\t'
        f'{row["n_mrca_descendant_leaves"]}\t{row["n_extra_descendant_leaves"]}\t'
        f'{row["forms_exclusive_clade"]}\t{row["mrca_is_root"]}'
    )

print("\nOutput:", out_tsv)
print("Note: Support values are FastTree SH-like local support values, not conventional bootstrap values.")
print("When the MRCA is the root, support is reported as NA because the root has no parent branch to evaluate.")

if Path("/content").exists():
    from google.colab import files
    files.download(str(out_tsv))